In [ ]:
data\market_data.py
import yfinance as yf
import pandas as pd
from datetime import datetime
from data.option_filters import liquid_options

def year_fraction(maturity: str) -> float:
    today = datetime.today()
    expiry = datetime.strptime(maturity, "%Y-%m-%d")
    return (expiry - today).days / 365.0

def get_all_options(ticker, spread_limit):
    tk = yf.Ticker(ticker)
    spread = spread_limit
    spot = tk.history(period='1d')['Close'].iloc[-1]
    maturities = tk.options

    all_data=[]

    for maturity in maturities:
        T= year_fraction(maturity)

        chain = tk.option_chain(maturity)

        calls = liquid_options(chain.calls, spread)
        puts = liquid_options(chain.puts, spread)

        calls['type'] = 'call'
        puts['type'] = 'put'

        df = pd.concat([calls, puts])
        df["maturity"] = maturity
        df["T"] = T
        df["spot"] = spot
        df["ticker"] = ticker
        df['ExerciseStyle'] = 'american'

        all_data.append(df)
    return pd.concat(all_data, ignore_index=True)
    

def get_multiple_tickers(tickers, spread_limit):
    all_tickers = []

    for ticker in tickers:
        print(f'pulling...{ticker}...')
        df = get_all_options(ticker, spread_limit)
        all_tickers.append(df)
    return pd.concat(all_tickers, ignore_index=True)

data\option_filters.py
import pandas as pd

def liquid_options(df: pd.DataFrame, spread_limit=0.05):
    # Removing completely illiquid options
    df_volume = df[df.volume > 0].copy()

    # Calculating mid price
    mid = (df_volume.ask + df_volume.bid) / 2

    # Removing wide bid-ask spreads (relative to mid price)
    df_final = df_volume[(df_volume.ask - df_volume.bid) / mid < spread_limit]

    return df_final

simulation\heston_path.py
# ---------------------------
# 1️⃣ Heston Path Simulations
# ---------------------------

import numpy as np
def simulate_heston_paths_american_put_without_dividends(S0, r, T, v0, kappa, theta, sigma, rho, M, N):
    """Simulate Heston paths without dividends (American Put pricing)"""
    dt = T / M
    S = np.zeros((N, M+1))
    v = np.zeros((N, M+1))
    S[:,0] = S0
    v[:,0] = v0

    Z1 = np.random.normal(size=(N, M))
    Z2 = np.random.normal(size=(N, M))
    W1 = np.sqrt(dt) * Z1
    W2 = np.sqrt(dt) * (rho * Z1 + np.sqrt(1 - rho**2) * Z2)
    
    
    for t in range(M):
        #Variance discretization process - Euler discretization
        # Full truncation Euler for variance
        v[:, t+1] = np.maximum(
            v[:, t]
            + kappa * (theta - v[:, t]) * dt
            + sigma * np.sqrt(np.maximum(v[:, t], 0)) * W2[:, t],
            0
        )
        # Log-Euler for stock
        S[:, t+1] = S[:, t] * np.exp(
            (r - 0.5 * v[:, t]) * dt
            + np.sqrt(np.maximum(v[:, t], 0)) * W1[:, t]
        )
    return S, v, dt


def simulate_heston_paths_american_put_with_dividends(S0, r, T, v0, kappa, theta, sigma, rho, M, N, q):
    """
    Simulate Heston paths under risk-neutral measure with continuous dividends.

    Parameters:
        q : continuous dividend yield

    Returns:
        S : stock paths (N x M+1)
        v : variance paths (N x M+1)
        dt : time step
    """
    dt = T / M
    S = np.zeros((N, M+1))
    v = np.zeros((N, M+1))
    S[:,0] = S0
    v[:,0] = v0

    Z1 = np.random.normal(size=(N, M))
    Z2 = np.random.normal(size=(N, M))
    W1 = np.sqrt(dt) * Z1
    W2 = np.sqrt(dt) * (rho * Z1 + np.sqrt(1 - rho**2) * Z2)
    
    
    for t in range(M):
        #Variance discretization process - Euler discretization
        # Full truncation Euler for variance
        v[:, t+1] = np.maximum(
            v[:, t]
            + kappa * (theta - v[:, t]) * dt
            + sigma * np.sqrt(np.maximum(v[:, t], 0)) * W2[:, t],
            0
        )
        #only stock's drift part will change
        # Log-Euler for stock 
        S[:, t+1] = S[:, t] * np.exp(
            (r - q- 0.5 * v[:, t]) * dt
            + np.sqrt(np.maximum(v[:, t], 0)) * W1[:, t]
        )
    return S, v, dt
    


def simulate_heston_paths_american_call_with_dividends(S0, r, T, v0, kappa, theta, sigma, rho, M, N, q):
    """Simulate Heston paths with dividend yield q (American Call pricing)"""
    dt = T / N
    S = np.zeros((M, N+1))
    v = np.zeros((M, N+1))
    S[:,0] = S0
    v[:,0] = v0

    Z1 = np.random.normal(size=(M, N))
    Z2 = np.random.normal(size=(M, N))
    W1 = np.sqrt(dt) * Z1
    W2 = np.sqrt(dt) * (rho*Z1 + np.sqrt(1 - rho**2)*Z2)

    for t in range(N):
        v[:,t+1] = np.maximum(v[:,t] + kappa*(theta - v[:,t])*dt + sigma*np.sqrt(v[:,t])*W2[:,t], 0)
        S[:,t+1] = S[:,t] * np.exp((r - q - 0.5*v[:,t])*dt + np.sqrt(v[:,t])*W1[:,t])
    return S, v, dt

simulation\lsmc.py
# ---------------------------
# 2️⃣ LSMC(Longstaff–Schwartz regression) American Option Pricing 
# ---------------------------
import numpy as np

#Applicable for american put (with or w/o dividends)
def american_put_lsmc_vec(S, K, r, dt):
    N, M = S.shape
    M -= 1
    #Terminal payoff
    cashflow = np.maximum(K - S[:,-1], 0)

    #backward induction
    for t in range(M-1, 0, -1):
        X = S[:,t]
        #identify in-the-money paths
        itm = X < K
        if np.sum(itm) == 0:
            #only ITM paths matter for exercise decisions
            cashflow *= np.exp(-r*dt)
            continue
        #discount continuation value
        Y = cashflow * np.exp(-r*dt)
        #regression step (in the future, include v_t also for regression)
        coeffs = np.polyfit(X[itm], Y[itm], 2)
        #computing continuation value
        C = np.polyval(coeffs, X[itm])
        #exercise decision
        #exercise if : Immediate payoff > Continuation value
        exercise = (K - X[itm]) > C

        #update cashflow
        cashflow[itm] = np.where(exercise, K - X[itm], Y[itm])
        cashflow[~itm] *= np.exp(-r*dt)

        #discount to time 0
    return np.mean(cashflow * np.exp(-r*dt))

#Applicable for american call (with or w/o dividends)
def american_call_lsmc_vec(S, K, r, dt):
    N, M = S.shape
    M -= 1
    #Terminal payoff
    cashflow = np.maximum(S[:,-1] - K, 0)

    #backward induction
    for t in range(M-1, 0, -1):
        X = S[:,t]
        #identify in-the-money paths (counting number of paths in the money)
        itm = X > K
        if np.sum(itm) == 0: #if all paths are out-of-the-money--> no early exercise possible
            #we just discount the existing cashflows one time step back
            #only ITM paths matter for exercise decisions
            cashflow *= np.exp(-r*dt)
            continue # continue --> skip the regression and exercise logic fof this time step
        
        Y = cashflow * np.exp(-r*dt)
        coeffs = np.polyfit(X[itm], Y[itm], 2)
        C = np.polyval(coeffs, X[itm])
        exercise = (X[itm] - K) > C
        cashflow[itm] = np.where(exercise, X[itm] - K, Y[itm])
        cashflow[~itm] *= np.exp(-r*dt)
    return np.mean(cashflow * np.exp(-r*dt))

pricing\american.py
import numpy as np
from typing import Literal

#import functions for american puts/calls
from simulation.heston_path import simulate_heston_paths_american_put_without_dividends, simulate_heston_paths_american_put_with_dividends, simulate_heston_paths_american_call_with_dividends
#american call without dividends is same as european call
from models.heston_european import heston_european_call_price

from simulation.lsmc import american_put_lsmc_vec, american_call_lsmc_vec

def american_put_without_dividends(S0, K, r,T, v0, kappa, theta, sigma, rho, M, N):
    """
    Wrapper function that:
    1. Simulates Heston paths
    2. Applies LSMC
    """
    S, v, dt = simulate_heston_paths_american_put_without_dividends( S0, r, T, v0, kappa, theta, sigma, rho, M, N)

    american_put_price_without_dividends = american_put_lsmc_vec(S, K, r, dt)

    return american_put_price_without_dividends 

def american_put_with_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho, M, N, q):
    """
    Wrapper function that:
    1. Simulates Heston paths
    2. Applies LSMC
    """
    S, v, dt = simulate_heston_paths_american_put_with_dividends( S0, r, T,v0, kappa, theta, sigma, rho, M, N, q)

    american_put_price_with_dividends = american_put_lsmc_vec(S, K, r, dt)

    return american_put_price_with_dividends

#American call options without dividends (same as european call price)
def american_call_without_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho):

    params = (v0 ,kappa, theta, sigma, rho)

    american_call_price_without_dividends = heston_european_call_price(
        S0=S0,
        K=K,
        r=r,
        T=T,
        params=params,
    )
    return american_call_price_without_dividends



def american_call_with_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho, M, N, q):

    # 1️⃣ Simulate paths
    S, v, dt = simulate_heston_paths_american_call_with_dividends(S0, r, T, v0, kappa, theta, sigma, rho, M, N, q
    )
    #pricing from lsmc
    american_call_price_with_dividends = american_call_lsmc_vec(S, K, r, dt)
    return american_call_price_with_dividends

pricing\european.py
from models.heston_european import heston_european_call_price
from models.heston_european import heston_european_put_price

def heston_european_call_option(S0, K, r, T, v0, kappa, theta, sigma, rho):
    """
    Wrapper for European option pricing under Heston.

    Parameters
    ----------
    S0 : float
        Initial stock price
    K : float
        Strike
    r : float
        Risk-free rate
    T : float
        Maturity
    kappa, theta, sigma, rho, v0 : Heston parameters

    Returns
    -------
    price : float
    """
    params = (v0, kappa, theta, sigma, rho)

    european_call_price = heston_european_call_price(
        S0=S0,
        K=K,
        r=r,
        T=T,
        params=params
    )

    return european_call_price

def heston_european_put_option(S0, K, r, T, v0, kappa, theta, sigma, rho):
    
    params = (kappa, theta, sigma, rho, v0)

    european_put_price = heston_european_put_price(
        S0=S0,
        K=K,
        r=r,
        T=T,
        params=params
    )

    return european_put_price

models\heston_cf.py
import numpy as np
from scipy.integrate import quad


def heston_cf(u, params, S0, r, T, j):

    v0, kappa, theta, sigma, rho = params
    x = np.log(S0)

    if j == 1:
        b = kappa - rho*sigma
        u_bar = 0.5
    else:
        b = kappa
        u_bar = -0.5

    d = np.sqrt((rho*sigma*1j*u - b)**2 - sigma**2*(2*u_bar*1j*u - u**2))
    g = (b - rho*sigma*1j*u + d) / (b - rho*sigma*1j*u - d)

    C = r*1j*u*T + (kappa*theta/sigma**2)*(
        (b - rho*sigma*1j*u + d)*T
        - 2*np.log((1 - g*np.exp(d*T)) / (1 - g))
    )

    D = (b - rho*sigma*1j*u + d)/sigma**2 * (
        (1 - np.exp(d*T)) / (1 - g*np.exp(d*T))
    )

    return np.exp(C + D*v0 + 1j*u*x)
models\heston_european.py
import numpy as np
from scipy.integrate import quad
from models.Heston_cf import heston_cf



def P_integral(params, S0, K, r, T, j):
    integrand = lambda u: np.real(np.exp(-1j*u*np.log(K)) * heston_cf(u, params, S0, r, T, j) / (1j*u))
    return 0.5 + (1/np.pi) * quad(integrand, 0, 100)[0]



def heston_european_call_price(S0, K, r, T, params):
    P1 = P_integral(params, S0, K, r, T, 1)
    P2 = P_integral(params, S0, K, r, T, 2)
    #call
    return S0*P1 - K*np.exp(-r*T)*P2

def heston_european_put_price(S0, K, r, T, params):
    P1 = P_integral(params, S0, K, r, T, 1)
    P2 = P_integral(params, S0, K, r, T, 2)
    #put
    return K*np.exp(-r*T)*(1-P2) - S0*(1-P1)
models\black_scholes.py
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq


def black_scholes_price(S, K, r, T, sigma, option_type, q):
    """
    European Black-Scholes price with continuous dividend yield.
    """
    if T <= 0:
        if option_type.lower() == "call":
            return max(S - K, 0.0)
        elif option_type.lower() == "put":
            return max(K - S, 0.0)
        else:
            raise ValueError("option_type must be 'call' or 'put'")

    d1 = (np.log(S/K) + (r - q + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    if option_type.lower() == "call":
        return S*np.exp(-q*T)*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

    elif option_type.lower() == "put":
        return K*np.exp(-r*T)*norm.cdf(-d2) - S*np.exp(-q*T)*norm.cdf(-d1)

    else:
        raise ValueError("option_type must be 'call' or 'put'")

calibration/implied_vol.py
import numpy as np
from scipy.optimize import brentq
from models.black_scholes import black_scholes_price

"""
Since there is no algebraic way to rearrange the Black-Scholes equation to solve for sigma, 
we use a numerical root-finding algorithm (like Newton-Raphson or Brent's method) to find the volatility 
that makes the Black-Scholes price equal to our Heston model price. Here we are using Brent's method.
"""

def implied_volatility(heston_model_price, S, K, r, T, option_type, q):

    if heston_model_price <= 0 or T <= 0:
        return np.nan
    
    def objective(sigma):
        return black_scholes_price(
            S, K, r,T, sigma, option_type, q
        ) - heston_model_price

    try:
        iv = brentq(objective, 1e-6, 5.0)
        return iv
    except ValueError:
        return np.nan
    
calibration/heston_loss_function.py
import numpy as np
from scipy.optimize import minimize
from pricing.american import american_put_without_dividends, american_put_with_dividends, american_call_without_dividends, american_call_with_dividends
from pricing.european import heston_european_call_option, heston_european_put_option

def heston_loss(params, r, options_df, M, N, q):
    v0, kappa, theta, sigma, rho = params

    #variance should be positive
    if any (p<0 for p in [v0, kappa, theta, sigma]) or not (-1 < rho< 1):
        return 1e10
    
    errors = []

    for idx, row in options_df.iterrows():
        S0= row['spot']
        K = row['strike']
        T = row['T']
        r = r
        market_price = row['lastPrice'] #or mid price
        #options_type = row['type']
        exercise_style = row['ExerciseStyle']

        if exercise_style.lower() == 'european' and market_price >0:
                if row['type'] == 'call':
                    model_price = heston_european_call_option(S0, K, r,T, v0, kappa, theta, sigma, rho)
                elif row['type']=='put':
                    model_price = heston_european_put_option(S0, K, r, T, v0, kappa, theta, sigma, rho)
                else:
                    raise ValueError("European option_type must be 'call' or 'put'")
            
        elif exercise_style.lower()=='american' and market_price >0:
            if row['type'] == 'call' and q ==0:
                model_price = american_call_without_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho)
            elif row['type']=='call' and q != 0:
                model_price = american_call_with_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho, M, N, q)
            elif row['type']== 'put' and q==0:
                model_price = american_put_without_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho, M, N)
            elif row['type'] == 'put' and q!=0:
                model_price = american_put_with_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho, M, N, q)
            else:
                raise ValueError("American option_type must be 'call' or 'put'")
        else: 
            raise ValueError("Exercise style is not european or american")
        
        errors.append((model_price - market_price)**2)
    return np.sum(errors)

calibration/calibrate_heston.py
import numpy as np
from scipy.optimize import minimize
from calibration.heston_loss_function import heston_loss

def calibrate_heston(options_df, r, M, N, q, initial_guess, bounds=None):
    if bounds is None:
        bounds=[(1e-4, 2), #v0
                (1e-4, 10), #kappa
                (1e-4, 2), #theta
                (1e-4, 2), #sigma
                (-0.999, 0.999)] #rho
    
    result = minimize(
        heston_loss, 
        x0=initial_guess, 
        args=(r, options_df, M, N, q),
        bounds = bounds,
        method='L-BFGS-B',
        options={'disp': True, 'maxiter': 500}
    )
    
    return result.x, result.fun

main.ipynb
import sys
import os
sys.path.append(os.path.abspath(".."))
import numpy as np
import pandas as pd
from data.market_data import get_multiple_tickers
from pricing.american import american_put_without_dividends, american_put_with_dividends, american_call_without_dividends, american_call_with_dividends
from pricing.european import heston_european_call_option, heston_european_put_option
from calibration.implied_vol import implied_volatility
tickers = ['AAPL', 'NVDA']
r = 0.05
q=0
N= 10000
M = 100
spread_limit = 0.05
options_df = get_multiple_tickers(tickers, spread_limit=spread_limit)
options_df_nvda = options_df[(options_df['ticker']=='NVDA') & (options_df['maturity']=='2026-04-17') &(options_df['strike'] == 200) & (options_df['type']=='call')]
heston_params = (0.04, 2.0, 0.04, 0.5, -0.7)
def price_row(row, r, q, heston_params ):
    S0 = row['spot']
    K=row['strike']
    T= row['T']
    option_type = row['type']

    v0, kappa, theta, sigma, rho = heston_params

    if option_type == 'call':
        if q==0:
            print('american_call_without_dividends ....exexcuted')
            return american_call_without_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho)
        else:
            print('american_call_with_dividends....exexcuted')
            return american_call_with_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho, M, N, q)
    elif option_type == 'put':
        if q==0:
            print('american_put_without_dividends....exexcuted')
            return american_put_without_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho, M, N)
        else:
            print('american_put_with_dividends....exexcuted')
            return american_put_with_dividends(S0, K, r, T, v0, kappa, theta, sigma, rho, M, N, q)
options_df_nvda['heston_price'] = options_df_nvda.apply(lambda row: price_row(row, r, q, heston_params), axis=1)

options_df_nvda["market_iv"] = options_df_nvda.apply(
    lambda row: implied_volatility(
        row["heston_price"],
        row["spot"],
        row["strike"],
        r,
        row["T"],
        row["type"],
        q
    ),
    axis=1
)
options_df_nvda.head()
#Heston Model Calibration
calibration_df_nvda = options_df[options_df['ticker']=='NVDA']
from calibration.calibrate_heston import calibrate_heston
r = 0.05
q=0
N= 10000
M = 100
initial_guess = [
    0.04,   # v0  (initial variance ≈ 20% vol)
    2.0,    # kappa
    0.04,   # theta
    0.5,    # sigma
    -0.7    # rho
]

params_opt, loss_val = calibrate_heston(
    options_df=calibration_df_nvda,
    r=r,
    M=M,
    N=N,
    q=q,
    initial_guess=initial_guess
)

print("Calibrated Parameters:")
print("v0    =", params_opt[0])
print("kappa =", params_opt[1])
print("theta =", params_opt[2])
print("sigma =", params_opt[3])
print("rho   =", params_opt[4])
print("\nFinal Loss =", loss_val)

NameError: name 'market_data' is not defined